# F5-TTS Zero-Shot Voice Cloning

Clone any voice from a short reference WAV file. Free T4 GPU.
**No training needed.** Just upload a reference recording and type text.

---
### Instructions
1. Runtime → Change runtime type → **T4 GPU**
2. Run Cell 1 to mount Google Drive
3. Upload your reference WAV to `MyDrive/rio_voice/reference/` in Google Drive
4. Run all cells in order
5. Download the generated WAV files from the Files panel on the left

In [ ]:
# CELL 1: Mount Google Drive & install F5-TTS
from google.colab import drive
drive.mount("/content/drive")

!pip install -q git+https://github.com/SWivid/F5-TTS.git soundfile

In [ ]:
# CELL 2: Set paths and select reference audio
from pathlib import Path

# --- REFERENCE AUDIO ---
# Put your reference WAV file in Google Drive at:
#   MyDrive/rio_voice/reference/
# Best choices from your recordings:
#   lk_expressive.wav  (English, 65s, expressive voice)
#   lk_intro.wav       (English, clean intro)
#   lk_natural.wav     (English, natural speech)
#
# Change REFERENCE_NAME to your actual file name:

REFERENCE_NAME = "lk_expressive.wav"  # <--- CHANGE THIS
REFERENCE_DIR = Path("/content/drive/MyDrive/rio_voice/reference")
OUTPUT_DIR = Path("/content/drive/MyDrive/rio_voice/output")

REFERENCE_WAV = REFERENCE_DIR / REFERENCE_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not REFERENCE_WAV.exists():
    # Try normalized version
    alt_name = REFERENCE_NAME.replace("lk_", "lk_").replace(".wav", ".wav")
    alt_path = REFERENCE_DIR / alt_name
    if alt_path.exists():
        REFERENCE_WAV = alt_path
    else:
        # List available files
        print("Available reference files:")
        for f in sorted(REFERENCE_DIR.glob("*.wav")):
            print(f"  {f.name}")
        raise FileNotFoundError(f"Reference file not found: {REFERENCE_WAV}\n"
                                f"Upload a WAV file to {REFERENCE_DIR}")

print(f"Reference: {REFERENCE_WAV.name}")
print(f"Output:    {OUTPUT_DIR}")

In [ ]:
# CELL 3: Load F5-TTS model
import torch
import soundfile as sf
import torchaudio

# Shim to avoid audio backend issues in Colab
def _sf_load(path, *args, **kwargs):
    data, sr = sf.read(str(path), dtype="float32", always_2d=True)
    return torch.from_numpy(data.T), sr

torchaudio.load = _sf_load

print("Loading F5-TTS...")
from f5_tts.api import F5TTS
f5 = F5TTS()
print("F5-TTS loaded successfully!")

In [ ]:
# CELL 4: Generate demo sentences
# The reference text should be what was actually said in the reference audio.
# By default we use the intro from the script. Edit REFERENCE_TEXT if needed.

REFERENCE_TEXT = (
    "Hello, good morning. I hope you are doing well today. "
    "My name is Ashwini, and I am calling from Yexis Electronics. "
    "We are regional distributors for Samsung products, including commercial displays, "
    "signage boards, video walls, monitors, televisions, and air conditioning solutions."
)

# --- Demo sentences to generate ---
DEMO_TEXTS = [
    # English - sales call intro
    "Hello, this is Ashwini calling from Yexis Electronics. "
    "I wanted to understand your display requirement for the office.",

    # English - follow-up
    "Thank you for your time. I will send the quotation by today evening. "
    "Please let me know if you have any questions.",

    # English - confident pitch
    "For commercial usage, I would recommend the Samsung professional display. "
    "It is more reliable for long hours of operation.",
]

print(f"Reference text ({len(REFERENCE_TEXT)} chars):")
print(f"  {REFERENCE_TEXT[:100]}...")
print(f"\nDemo sentences to generate: {len(DEMO_TEXTS)}")
for i, t in enumerate(DEMO_TEXTS):
    print(f"  {i+1}. {t[:80]}...")

In [ ]:
# CELL 5: RUN INFERENCE
# This takes ~2-5 minutes per sentence on T4 GPU

from IPython.display import Audio, display

generated_files = []

for i, gen_text in enumerate(DEMO_TEXTS):
    out_path = OUTPUT_DIR / f"demo_{i+1}.wav"
    print(f"\n[{i+1}/{len(DEMO_TEXTS)}] Generating: {out_path.name}")
    print(f"  Text: {gen_text[:80]}...")

    f5.infer(
        ref_file=str(REFERENCE_WAV),
        ref_text=REFERENCE_TEXT,
        gen_text=gen_text,
        file_wave=str(out_path),
        show_info=print,
    )

    generated_files.append(out_path)
    print(f"  -> Saved: {out_path}")
    display(Audio(str(out_path)))

print(f"\n✅ ALL DONE! Generated {len(generated_files)} files in:")
print(f"   {OUTPUT_DIR}")
for f in generated_files:
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"   - {f.name} ({size_mb:.2f} MB)")

In [ ]:
# CELL 6: Try Hindi with the same voice
# F5-TTS can do multilingual with the same reference!

print("Generating Hindi demo...")

hindi_text = (
    "Namaste sir, main Yexis Electronics se Ashwini bol rahi hoon. "
    "Aapki display requirement samajhna chahti hoon."
)

hindi_out = OUTPUT_DIR / "demo_hindi.wav"
f5.infer(
    ref_file=str(REFERENCE_WAV),
    ref_text=REFERENCE_TEXT,
    gen_text=hindi_text,
    file_wave=str(hindi_out),
    show_info=print,
)

print(f"\nSaved: {hindi_out}")
display(Audio(str(hindi_out)))

In [ ]:
# CELL 7: Download all generated files as a ZIP
!cd "{OUTPUT_DIR}" && zip -j /content/f5_tts_output.zip *.wav
from google.colab import files
files.download("/content/f5_tts_output.zip")
print("\nDownload started! If it doesn't auto-download, check the Files panel on the left.")

### Quick Reference

**To use a different reference file:**
1. Upload your WAV to `MyDrive/rio_voice/reference/` in Google Drive
2. Edit `REFERENCE_NAME` in Cell 2
3. Edit `REFERENCE_TEXT` in Cell 4 to match what's actually said in that WAV
4. Run Cells 4 onwards again

**To generate your own text:**
1. Edit the `DEMO_TEXTS` list in Cell 4
2. Run Cells 5 onwards

**Troubleshooting:**
- **OOM (out of memory)**: Generate fewer sentences at a time, or use a shorter reference clip
- **Bad quality**: Use a cleaner reference recording (minimal background noise)
- **Wrong accent**: Make sure REFERENCE_TEXT matches what the person actually said in the WAV